## RAG Pipeline - Data Ingestion to Vector DB

In [3]:
import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/3d/n1gdt7t127l2b51qkg822cpc0000gn/T/ipykernel_77796/2234884931.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [31]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory: str):
    """Process all the pdfs in a directory"""
    
    # list to store all loaded docs
    all_docs = []

    pdf_dir = Path(pdf_directory)

    #Find all PDF Recursively
    #.glob is to find file names matcing the pdf pattern, **/ means “also look through subfolders,
    # so it finds PDFs inside data/pdf and its nested folders.
    pdf_files = list(pdf_dir.glob("**/*.pdf")) # list -> [ Path("/dir/example.pdf") , Path("/dir/example2.pdf") ]
    print(f"Found {len(pdf_files)} PDF files to be processed")

    for file in pdf_files:
        print(f"\n Processing{file.name}") #.name is a property of Path object -> example.pdf from the /dir/example.pdf path 

        try: 
            loader = PyPDFLoader(str(file))
            documents = loader.load() #PDFLoader.load() returns a list of LangChain Document objects, usually one Document per PDF page.

            #this is the way to extract any suffix, we could have just use the string "pdf" as well in metadata
            fileType = file.suffix
            fileTypeStr = fileType.lstrip(".")

            # Add Source Info to Metadata
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = fileTypeStr

            all_docs.extend(documents) # eg: ["file1"] and ["file2", "file3"] becomes  ["file1", "file2", "file3"] if we use .append it becomes ["file1", "file2", "file3"]
            print(f" ✅ Loaded {len(documents)} Pages")

        except Exception as e:
            print(f" ❌ Error: {e}")
            
    print(f"\n Total Documents Loaded {len(all_docs)}")
    return all_docs

all_pdf_documents = process_all_pdfs("data")

    

Found 2 PDF files to be processed

 Processingnorthstar_support_policy.pdf
 ✅ Loaded 1 Pages

 Processingvector_search_field_guide.pdf
 ✅ Loaded 1 Pages

 Total Documents Loaded 2


In [37]:
# text splitting in chunks 


def split_documents(loaded_docs, chunk_size = 1000, chunk_overlap=200):
    """Splitting Text in smaller chunks for better RAG Performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""] #
    )

    split_docs = text_splitter.split_documents(loaded_docs)
    print(f"Spliting {len(loaded_docs)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example Chunk:")
        print(f"Content: {split_docs[0].page_content[:100]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [38]:
chunks = split_documents(all_pdf_documents)
chunks

Spliting 2 documents into 8 chunks

 Example Chunk:
Content: RAG learning sample | Fictional training material
Page 1
SAMPLE DOCUMENT 01 / CUSTOMER SUPPORT POLIC...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-09T01:43:02-07:00', 'author': 'RAG Learning Samples', 'keywords': '', 'moddate': '2026-11-09T01:43:02-07:00', 'subject': '(unspecified)', 'title': 'Northstar Learning: Support Policy', 'trapped': '/False', 'source': 'data/pdf/northstar_support_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'northstar_support_policy.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-09T01:43:02-07:00', 'author': 'RAG Learning Samples', 'keywords': '', 'moddate': '2026-11-09T01:43:02-07:00', 'subject': '(unspecified)', 'title': 'Northstar Learning: Support Policy', 'trapped': '/False', 'source': 'data/pdf/northstar_support_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'northstar_support_policy.pdf', 'file_type': 'pdf'}, page_content='RAG learning sample | Fictional training material\nPage 1\nSAMPLE DOCUMENT 01 / CUSTOMER SUPPORT POLICY\nNorthstar Learning: Support Policy\nNorthstar Learning is a fictional online course provider. This document is a sample policy for practicing PDF loading,\nchunking, metadata capture, and retrieval. The dates, amounts, and contact details below are invented for this exercise.\nSupport hours and response times\nThe support desk is open Monday through Friday, 9:00 a.m. to 5:00 p.m. 